In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.options.plotting.backend = "plotly"
pd.options.display.float_format = lambda x: f"{x:,.4f}"

from utils.logger import get_logger
from loaders.bdf_api import get_bdf_series
from loaders.sg_pee import get_sg_pee_data
from config.settings import (
    BDF_BASE_URL,
    BDF_HEADERS,
    KEY_INFLATION,
    KEY_LIVRET_A,
    ISIN_PEE,
    DRIVER_PATH,
)

from core.accounts.regulated_savings_account_v2 import RegulatedSavingAccount
from core.accounts.listed_account_v2 import ListedAccount

logger = get_logger("Main")

In [ ]:
def ensure_cache_dir(path: str = "cache"):
    """Ensure cache directory exists."""
    if not os.path.exists(path):
        os.makedirs(path)
        logger.info(f"Created cache directory at '{path}'")


def main():
    logger.info("=== Starting financial data load ===")
    ensure_cache_dir()

    data_sources = {
        "Livret A": {
            "func": get_bdf_series,
            "params": {
                "series_key": KEY_LIVRET_A,
                "base_url": BDF_BASE_URL,
                "headers": BDF_HEADERS,
                "start_date": "2020-01-01",
                "cache_path": "cache/livret_a.pkl",
            },
        },
        "Inflation": {
            "func": get_bdf_series,
            "params": {
                "series_key": KEY_INFLATION,
                "base_url": BDF_BASE_URL,
                "headers": BDF_HEADERS,
                "start_date": "2020-01-01",
                "cache_path": "cache/inflation.pkl",
            },
        },
        "PEE": {
            "func": get_sg_pee_data,
            "params": {
                "isin": ISIN_PEE,
                "driver_path": DRIVER_PATH,
                "headless": True,
                "cache_path": "cache/pee.pkl",
            },
        },
    }

    results = {}

    for label, info in data_sources.items():
        try:
            logger.info(f"Loading {label} data...")
            df = info["func"](**info["params"])
            logger.info(f"{label} data loaded successfully ({len(df)} rows)")
            results[label] = df
        except Exception as e:
            logger.error(f"Failed to load {label} data: {e}")
            results[label] = None

    logger.info("=== Financial data load completed ===")

    return results




In [ ]:
'''
def clear_cache(cache_name=None):
    """Supprime le cache. Si cache_name=None, supprime tous les caches."""
    cache_dir = "cache"
    if cache_name:
        path = os.path.join(cache_dir, cache_name)
        if os.path.exists(path):
            os.remove(path)
            logger.info(f"Cache '{cache_name}' supprimé")
        else:
            logger.info(f"Cache '{cache_name}' non trouvé")
    else:
        for f in os.listdir(cache_dir):
            if f.endswith(".pkl"):
                os.remove(os.path.join(cache_dir, f))
        logger.info("Tous les caches ont été supprimés")

clear_cache("pee.pkl")   # Supprime uniquement le cache PEE
clear_cache()            # Supprime tous les caches
'''


In [ ]:

results = main()
df_la = results['Livret A']
df_i = results['Inflation']
df_pee = results['PEE']


In [ ]:
df_data = pd.read_excel("data/investment.xlsx", index_col=[1, 2, 0], usecols=range(11))
today = datetime.today()
df_data[['other_fees', 'broker_fees', 'tax']] = df_data[['other_fees', 'broker_fees', 'tax']].fillna(0)
df_data['empty_date'] = pd.NaT # Date ou l'ensemble de la ligne a été soldée
df_data.sample(2)

In [ ]:
dfl = (
    df_data.loc['LR'][["value", "empty_date", "indice"]]
    .sort_index(axis=0)
    .copy()
)
dfl["value_adj"] = dfl["value"]
dfl[["rl_interest_adj", "th_interest_adj"]] = 0.0
dfl

"""
# dfl.groupby(level=0).apply(lambda g: FixedRateInvestment(g))
la = FixedRateInvestment(dfl.loc['Livret A'])
ld = FixedRateInvestment(dfl.loc['LDDS'])
lj = FixedRateInvestment(dfl.loc['Livret jeune'])
"""

In [ ]:
"""
dfl = (
    df_data.loc['LR'][["value", "empty_date", "indice"]]
    .sort_index(axis=0)
    .copy()
)
dfl["value_adj"] = dfl["value"]
dfl[["rl_interest_adj", "th_interest_adj"]] = 0.0
dfl

investments = dfl.groupby(level=0).apply(lambda g: FixedRateInvestment(g.name, g.droplevel(0)))
investments_timeline = pd.concat(investments.apply(lambda inv: inv.positions_timeline).to_dict(), axis=1)
investments_timeline.tail(3)
"""

In [ ]:
df_data.sample(3)

In [ ]:
df_data.loc['LR']

In [ ]:
df_data.loc['LR']

In [ ]:
df = df_data.loc['LR'].loc['Livret jeune'][["value", "empty_date", "indice"]].sort_index().copy()
df["value_adj"] = df["value"]
df[["rl_interest_adj", "th_interest_adj"]] = 0.0
df

In [ ]:
df_data.loc['LR'].loc['Livret A']

In [ ]:
lr = RegulatedSavingAccount(df_data.loc['LR'])

In [ ]:
lr.investments

In [ ]:
lr.investments_timeline.tail(4)

In [ ]:
lr.investments_timeline.iloc[-1].loc['Livret jeune'].unstack(1).iloc[:,:2].sum(axis=1)

In [ ]:
lr.investments_timeline.iloc[-1].xs('rl_interest_adj', level=2).groupby(level=0).sum()

In [ ]:
lr.investments_closed_positions

### Listed

In [ ]:
df_data_fs = pd.read_excel("data/investment.xlsx", sheet_name='rompu', index_col=[1,2,0])
df_data_fs.sample()


In [ ]:
df_data.loc['PEA']

In [ ]:
la = ListedAccount(df_data.loc['PEA'], df_data_fs.loc['PEA'], force_refresh=True)

In [ ]:
la.investments.iloc[0]._inflation_rates



In [ ]:
la.investments_closed_positions


In [ ]:
"""
from core.market_data.manager_v3 import MarketDataManager
md = MarketDataManager(list(df_data.loc['PEA'].index.get_level_values(0).unique().union(df_data.loc['PEA']['indice'].unique())))
                    
md = MarketDataManager(list(df_data.loc['PEA'].index.get_level_values(0).unique().union(df_data.loc['PEA']['indice'].unique())) + ['MSFT'])
md.get_market_data('AAPL')
"""

In [ ]:
aux = la.investments_timeline.loc[:, la.investments_timeline.columns.get_level_values(2) == 'count_nm']
aux = aux['AI.PA']
aux.columns = [
    '_'.join(map(str, col)).strip('_')
    for col in aux.columns
]
aux.plot()

In [ ]:
la.investments_timeline.head(3)

In [ ]:
la.investments_timeline.tail(3)

### Démonstration

In [ ]:
la.investments

In [ ]:
la.transactions

In [ ]:
la.investments_closed_positions

In [ ]:
la.investments_timeline.tail(3)

In [ ]:
la.tickers

In [ ]:
la.stock_data_fs

In [ ]:
target = 'AI.PA'

In [ ]:
la.investments.loc[target].market_data

In [ ]:
la.investments.loc[target].name

In [ ]:
la.investments.loc[target].transactions

In [ ]:
la.investments.loc[target].fractional_share_data

In [ ]:
la.investments.loc[target].positions_timeline.tail(3)

In [ ]:
la.investments.loc[target].metrics.keys()

In [ ]:
la.investments.loc[target].metrics['performance'].keys()

In [ ]:
pd.concat(la.investments.loc[target].metrics['positions']).unstack(0).tail(3)

In [ ]:
pd.concat(la.investments.loc[target].metrics['performance']).unstack(0).tail(3)

In [ ]:
pd.concat(la.investments.loc[target].metrics['performance']).unstack(0).iloc[-1].unstack(0)

In [ ]:
aux = pd.concat(
    [pd.Series(inv.metrics['risk']) * 100 for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux

In [ ]:
aux = pd.concat(la.investments.loc[target].metrics['positions']).unstack(0).iloc[-1].unstack(0)
aux

In [ ]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1, keys=[inv.name for inv in la.investments])

In [ ]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['equity_invested'].sort_index()

In [ ]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['equity_invested'].sum()

In [ ]:
from utils.fees import fees
((pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['equity_invested'].sort_index()).apply(fees) + pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['benefit'].sort_index())

In [ ]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['benefit'].sort_index()

In [ ]:
pd.concat([pd.concat(inv.metrics['performance']).unstack(0).iloc[-1].unstack(0) for inv in la.investments], axis=1).loc['benefit'].sum()

In [ ]:
la.investments_closed_positions

### Market value

In [ ]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['market_value'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux.div(aux.sum(axis=1), axis=0)
aux

In [ ]:
aux.sum(axis=1)

### Total returns

In [ ]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['total_returns'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux = aux * 100
aux

In [ ]:
# Aplatir les colonnes multi-index
aux_flat = aux.copy()
aux_flat.columns = ["_".join(map(str, col)) for col in aux_flat.columns]

# Maintenant tu peux plotter
aux_flat.plot()

### Annualized returns

In [ ]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['annualized_return'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux = aux * 100
aux

In [ ]:
aux_flat = aux.copy()
aux_flat.columns = ["_".join(map(str, col)) for col in aux_flat.columns]

for col in aux_flat.columns:

    mask = aux_flat[col].notna()

    if not mask.any():
        continue

    # position de la première valeur non-NaN
    start_pos = mask.values.argmax()

    # label d'index correspondant
    start_label = aux_flat.index[start_pos]

    # labels des 160 lignes suivantes
    end_pos = start_pos + 91
    end_pos = min(end_pos, len(aux_flat) - 1)
    end_label = aux_flat.index[end_pos]

    # mise à NaN avec loc
    aux_flat.loc[start_label:end_label, col] = np.nan

aux_flat.plot()

In [ ]:
la.market_data_manager._data['GLE.PA']

In [ ]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['annualized_return'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
aux * 100

In [ ]:
aux = pd.concat(
    [pd.concat(inv.metrics['performance']).loc['benefit'] for inv in la.investments],
    axis=1,
    keys = [inv.name for inv in la.investments]
)
# Aplatir les colonnes multi-index
aux_flat = aux.copy().ffill()
aux_flat.columns = ["_".join(map(str, col)) for col in aux_flat.columns]
aux_flat['Portfolio'] = aux_flat.sum(axis=1)
# Maintenant tu peux plotter
aux_flat.plot()

In [ ]:
aux_flat

In [ ]:
la.investments.iloc[2].metrics['risk_adjusted_performance']

In [ ]:
la.investments.iloc[2].transactions['annual_fee_rate'].values[0]

#.market_data['Close']

In [ ]:
i = 0
print(la.investments.iloc[i].name)
pd.concat(la.investments.iloc[i].metrics['performance']).unstack(0).xs('total_returns', axis=1, level=1).plot()

In [ ]:
la.investments.iloc[1].transactions

In [ ]:
la.investments.iloc[0].simulated_transactions

In [ ]:
la.investments.iloc[0].simulated_positions_timeline

In [ ]:
la.investments.iloc[1].simulated_transactions

In [ ]:
la.investments.iloc[1].simulated_positions_timeline

In [ ]:
la.investments.iloc[1].transactions

In [ ]:
df = la.investments.iloc[0].transactions
df

In [ ]:
hjn

In [ ]:
df_port = la.investments.iloc[0].positions_timeline
idx = pd.IndexSlice  # Pour manipuler MultiIndex proprement
df_port.loc[:, idx[:, 'count_nm']].droplevel(1, axis=1)

In [ ]:
def fees(x: float) -> float:
    """
    Returns the transaction fee charged by Bourse Direct for a PEA active_group_posount.

    The fee depends on the transaction amount based on predefined brackets:
    - Up to €500: €0.99
    - €501 to €1000: €1.90
    - €1001 to €2000: €2.90
    - €2001 to €4400: €3.80
    - Above €4400: 0.09% of the amount

    Args:
        x (float): The amount of the transaction (buy or sell)

    Returns:
        float: The corresponding fee in euros

    Examples:
        >>> fees(500)
        0.99
        >>> fees(1000)
        1.9
        >>> fees(2500)
        3.8
        >>> fees(6000)
        5.4
    """
    if x <= 500:
        return 0.99
    elif x <= 1000:
        return 1.9
    elif x <= 2000:
        return 2.9
    elif x <= 4400:
        return 3.80
    else:
        return x * 0.0009

df_port = la.investments.iloc[0].positions_timeline


In [ ]:
def compute_performance_metrics(
    self, 
    fees_func=None
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Compute historical performance metrics for this listed investment.

    This function calculates key time-series metrics per asset, including:
        - Nominal and adjusted share counts
        - Fractional shares
        - Market values and long values
        - Fees and taxes
        - Dividends
        - Equity invested
        - Asset-level returns and contribution to portfolio performance

    Parameters
    ----------
    fees_func : callable, optional
        Function to compute fees on market value. Signature: `fees_func(value: float) -> float`.
        If None, fees are assumed to be included in `fees_adj`.

    Returns
    -------
    tuple[pd.DataFrame, ...]
        df_nominal : Nominal share quantities ('count_nm')
        df_adjusted : Adjusted share quantities ('count_adj')
        df_fractional : Fractional shares ('fs_adj')
        df_long_value : Transaction-level value ('value_adj')
        df_fees : Fees paid ('fees_adj')
        df_taxes : Taxes paid ('tax_adj')
        df_market_value : Market value of positions
        df_equity : Equity invested (including fees/taxes)
        df_dividends : Cumulative dividends
        df_contribution : Asset-level performance contribution
        df_returns : Returns per asset (benefit / equity)
    """
    idx = pd.IndexSlice

    # Extract columns from investment timeline
    df_nominal = self.positions_timeline.loc[:, idx[:, "count_nm"]].droplevel(1, axis=1)
    df_adjusted = self.positions_timeline.loc[:, idx[:, "count_adj"]].droplevel(1, axis=1)
    df_fractional = self.positions_timeline.loc[:, idx[:, "fs_adj"]].droplevel(1, axis=1)
    df_long_value = self.positions_timeline.loc[:, idx[:, "value_adj"]].droplevel(1, axis=1)
    df_fees = self.positions_timeline.loc[:, idx[:, "fees_adj"]].droplevel(1, axis=1)
    df_taxes = self.positions_timeline.loc[:, idx[:, "tax_adj"]].droplevel(1, axis=1)

    # Market prices and dividends
    df_prices = self.stock_data["Close"]
    df_divs = self.stock_data["Dividends"].fillna(0.0)

    # Compute market value and equity invested
    df_market_value = df_adjusted.mul(df_prices, axis=0)
    df_equity = df_long_value * df_adjusted + df_fees + df_taxes

    # Compute cumulative dividends
    df_cumulative_divs = (df_nominal * df_divs).cumsum()

    # Compute total asset benefit
    if fees_func is None:
        fees_func = lambda x: 0.0  # assume fees are already in df_fees

    df_benefit = (
        df_market_value
        + df_cumulative_divs
        + df_fractional
        - df_market_value.applymap(fees_func)
        - df_equity
    )

    # Asset-level returns
    df_returns = df_benefit.div(df_equity.replace(0, np.nan))

    return (
        df_nominal,
        df_adjusted,
        df_fractional,
        df_long_value,
        df_fees,
        df_taxes,
        df_market_value,
        df_equity,
        df_cumulative_divs,
        df_benefit,
        df_returns,
    )


In [ ]:
import numpy as np

def compute_performance_metrics(
    self, 
    fees_func=None
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Compute historical performance metrics for this listed investment.

    This function calculates key time-series metrics per asset, including:
        - Nominal and adjusted share counts
        - Fractional shares
        - Market values and long values
        - Fees and taxes
        - Dividends
        - Equity invested
        - Asset-level returns and contribution to portfolio performance

    Parameters
    ----------
    fees_func : callable, optional
        Function to compute fees on market value. Signature: `fees_func(value: float) -> float`.
        If None, fees are assumed to be included in `fees_adj`.

    Returns
    -------
    tuple[pd.DataFrame, ...]
        df_nominal : Nominal share quantities ('count_nm')
        df_adjusted : Adjusted share quantities ('count_adj')
        df_fractional : Fractional shares ('fs_adj')
        df_long_value : Transaction-level value ('value_adj')
        df_fees : Fees paid ('fees_adj')
        df_taxes : Taxes paid ('tax_adj')
        df_market_value : Market value of positions
        df_equity : Equity invested (including fees/taxes)
        df_dividends : Cumulative dividends
        df_contribution : Asset-level performance contribution
        df_returns : Returns per asset (benefit / equity)
    """
    idx = pd.IndexSlice

    # Extract columns from investment timeline
    df_nominal = self.positions_timeline.loc[:, idx[:, "count_nm"]].droplevel(1, axis=1)
    df_adjusted = self.positions_timeline.loc[:, idx[:, "count_adj"]].droplevel(1, axis=1)
    df_fractional = self.positions_timeline.loc[:, idx[:, "fs_adj"]].droplevel(1, axis=1)
    df_long_value = self.positions_timeline.loc[:, idx[:, "value_adj"]].droplevel(1, axis=1)
    df_fees = self.positions_timeline.loc[:, idx[:, "fees_adj"]].droplevel(1, axis=1)
    df_taxes = self.positions_timeline.loc[:, idx[:, "tax_adj"]].droplevel(1, axis=1)

    # Market prices and dividends
    df_prices = self.stock_data["Close"]
    df_divs = self.stock_data["Dividends"].fillna(0.0)

    # Compute market value and equity invested
    df_market_value = df_adjusted.mul(df_prices, axis=0)
    df_equity = df_long_value * df_adjusted + df_fees + df_taxes

    # Compute cumulative dividends
    df_cumulative_divs = (df_nominal * df_divs).cumsum()

    # Compute total asset benefit
    if fees_func is None:
        fees_func = lambda x: 0.0  # assume fees are already in df_fees

    df_benefit = (
        df_market_value
        + df_cumulative_divs
        + df_fractional
        - df_market_value.applymap(fees_func)
        - df_equity
    )

    # Asset-level returns
    df_returns = df_benefit.div(df_equity.replace(0, np.nan))

    return (
        df_nominal,
        df_adjusted,
        df_fractional,
        df_long_value,
        df_fees,
        df_taxes,
        df_market_value,
        df_equity,
        df_cumulative_divs,
        df_benefit,
        df_returns,
    )


In [ ]:
"""
Compute historical portfolio and per-asset performance metrics, including returns, weights, dividends, and valuations.

Args:
    df_port (pd.DataFrame): Vectorized data of positions and metrics.

Returns:
    tuple of pd.DataFrames and pd.Series:
        - df_cn: Nominal share quantities.
        - df_c: Adjusted share quantities.
        - df_fs: Fractional shares.
        - df_lv: Long value of positions.
        - df_f: Fees.
        - df_t: Taxes.
        - df_p: Prices.
        - df_d: Dividends.
        - df_v: Market value.
        - df_w: Weights in portfolio.
        - df_e: Equity invested.
        - df_r: Returns per position.
        - df_benef: Value contribution per asset.
        - df_wr: Weighted returns.
        - portfolio_cum_return: Total cumulative return of portfolio.
"""

idx = pd.IndexSlice  # Pour manipuler MultiIndex proprement
df_cn = df_port.loc[:, idx[:, 'count_nm']].droplevel(1, axis=1)
df_c = df_port.loc[:, idx[:, 'count_adj']].droplevel(1, axis=1)
df_fs = df_port.loc[:, idx[:, 'fs_adj']].droplevel(1, axis=1)
df_lv = df_port.loc[:, idx[:, 'value_adj']].droplevel(1, axis=1) # Pour long value
df_f = df_port.loc[:, idx[:, 'fees_adj']].droplevel(1, axis=1)
df_t = df_port.loc[:, idx[:, 'tax_adj']].droplevel(1, axis=1)

df_p = la.investments.iloc[0].stock_data['Close']
df_d = la.investments.iloc[0].stock_data['Dividends']

df_v = df_cn.mul(df_p, axis=0)
df_e = df_lv*df_c + df_f + df_t
df_div = df_cn.mul(df_d, axis=0).cumsum()
df_benef = df_v + df_div + df_fs - df_v.apply(lambda stock: stock.apply(lambda val: fees(val))) - df_e
df_r = df_benef / df_e

df_w = df_v.div(df_v.sum(axis=1), axis=0)
df_wr = df_benef.div(df_e.sum(axis=1), axis=0)
portfolio_cum_return = df_wr.sum(axis=1)

df_r

In [ ]:
df_cn.mul(df_d, axis=0).cumsum()

### Niveau portefeuille

In [ ]:
df_w = df_v.div(df_v.sum(axis=1), axis=0)
df_wr = df_benef.div(df_e.sum(axis=1), axis=0)
portfolio_cum_return = df_wr.sum(axis=1)
portfolio_cum_return

In [ ]:
df_v = df_p * df_cn
df_w = df_v.div(df_v.sum(axis=1), axis=0)
df_e = df_lv*df_c + df_f + df_t
df_r = (df_v + (df_d*df_cn).cumsum() + df_fs - (df_v).apply(lambda stock: stock.apply(lambda val: fees(val))) - df_e) / df_e
df_benef = df_r.mul(df_e, axis=1)
df_wr = df_benef.div(df_e.sum(axis=1), axis=0)
portfolio_cum_return = df_wr.sum(axis=1)
df_benef

### Test

In [ ]:
from core.market_data.manager import MarketDataManager


In [ ]:
md = MarketDataManager(['AI.PA', 'AAPL', 'GLE.PA', 'AYV.PA', 'SU.PA', 'TEP.PA', 'DCAM.PA'])

In [ ]:
md.get_market_data('ESE.PA')

In [ ]:
df_data